[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/10_Image_Embeddings.ipynb)

# Embeddings de imágenes — CLIP

En **07** y **08** vimos embeddings de palabras (Word2Vec), y en **09** embeddings de texto con un modelo preentrenado (Gemini). Aquí damos un paso más: **una imagen también puede volverse un vector**.

Usamos **CLIP** (OpenAI, 2021): un modelo entrenado con 400 millones de pares imagen-texto de internet. Lo interesante no es solo que embebe imágenes — es que **mete texto e imágenes en el mismo espacio vectorial**. La foto de un gato y la frase "a photo of a cat" terminan con vectores parecidos.

Vamos en 5 pasos:

1. Cargar CLIP desde Hugging Face.
2. Embedding de una imagen.
3. Embedding de texto — mismo espacio, mismas dimensiones.
4. Comparar imagen y texto con similitud coseno.
5. Clasificación **zero-shot**: la imagen elige, entre varias frases, cuál la describe mejor — sin haber sido entrenada para esas categorías.

**Nota:** esta es una primera versión del notebook; la iremos afinando.

## Archivos que necesitas

Ninguno. Las imágenes de ejemplo se descargan solas desde Wikimedia Commons.

## Setup — instalar librerías

`transformers` trae CLIP listo para usar; no hace falta entrenar nada.

In [ ]:
%pip install -q transformers torch pillow numpy matplotlib requests

**GPU:** no es necesaria. El modelo (~150 MB) corre rápido en CPU. Si tienes GPU activada en Colab, se usa sola.

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})

## 0 — Imágenes de ejemplo

Ocho fotos de Wikimedia Commons. A propósito no son 8 categorías fáciles de separar:

- **bicicleta / motocicleta** — dos vehículos de dos ruedas, visualmente parecidos.
- **pizza / hamburguesa** — las dos son comida, y comparten colores y forma redonda.
- **hot dog** — trampa de nombre: la imagen es de comida, no de un perro, aunque la palabra "dog" esté ahí.
- **paisaje de montaña** — no le vamos a dar ninguna frase correcta en la sección 5. Sirve para ver qué hace el modelo cuando ninguna opción encaja.

In [ ]:
IMAGENES = {
    "gato": "https://commons.wikimedia.org/wiki/Special:FilePath/Cat03.jpg?width=400",
    "perro": "https://commons.wikimedia.org/wiki/Special:FilePath/Golde33443.jpg?width=400",
    "bicicleta": "https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle.jpg?width=400",
    "motocicleta": "https://commons.wikimedia.org/wiki/Special:FilePath/Motorcycle.jpg?width=400",
    "pizza": "https://commons.wikimedia.org/wiki/Special:FilePath/Eq_it-na_pizza-margherita_sep2005_sml.jpg?width=400",
    "hamburguesa": "https://commons.wikimedia.org/wiki/Special:FilePath/Cheeseburger.jpg?width=400",
    "hot dog (comida)": "https://commons.wikimedia.org/wiki/Special:FilePath/Chicago-style_hot_dog.jpg?width=400",
    "paisaje de montaña": "https://commons.wikimedia.org/wiki/Special:FilePath/Mountain_landscape.jpg?width=400",
}

# Wikimedia bloquea el User-Agent por defecto de requests (403 Forbidden);
# pide uno descriptivo — https://meta.wikimedia.org/wiki/User-Agent_policy
HEADERS_WIKIMEDIA = {"User-Agent": "IberoDiplomadoNLP/1.0 (uso educativo, sin fines comerciales)"}


def descargar_imagen(url):
    """Baja una imagen y la abre con PIL, sin guardarla a disco."""
    resp = requests.get(url, headers=HEADERS_WIKIMEDIA, timeout=20)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")

fotos = {nombre: descargar_imagen(url) for nombre, url in IMAGENES.items()}

fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax, (nombre, img) in zip(axes.flat, fotos.items()):
    ax.imshow(img)
    ax.set_title(nombre, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1 — Cargar CLIP

`openai/clip-vit-base-patch32`: un encoder de imágenes (Vision Transformer) y un encoder de texto, entrenados juntos para que la imagen y su descripción caigan cerca en el mismo espacio de 512 dimensiones.

In [ ]:
MODELO_CLIP = "openai/clip-vit-base-patch32"

modelo = CLIPModel.from_pretrained(MODELO_CLIP)
procesador = CLIPProcessor.from_pretrained(MODELO_CLIP)

print(f"Modelo listo: {MODELO_CLIP}")

## 2 — Embedding de una imagen

Le pasamos la foto del gato y obtenemos su vector de 512 números.

In [ ]:
entradas = procesador(images=fotos["gato"], return_tensors="pt")

with torch.no_grad():
    # transformers >= 5: get_image_features() regresa un objeto, no el vector
    # directo — el vector proyectado vive en .pooler_output
    vector_gato = modelo.get_image_features(**entradas).pooler_output[0]

print(f"Dimensiones del vector: {vector_gato.shape[0]}")
print(f"Primeros 8 valores: {vector_gato[:8].numpy().round(3)}")

## 3 — Embedding de texto (mismo espacio)

Ahora le pasamos frases en inglés — CLIP se entrenó sobre todo con captions en ese idioma, así que funciona mejor ahí. Son **7 frases para 8 imágenes**: a propósito no incluimos ninguna que describa el paisaje de montaña (sección 5). Fíjate que el vector de texto tiene **las mismas 512 dimensiones** que el de la imagen: viven en el mismo espacio.

In [ ]:
textos = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a bicycle",
    "a photo of a motorcycle",
    "a photo of a pizza",
    "a photo of a hamburger",
    "a photo of a hot dog",
]

entradas_texto = procesador(text=textos, return_tensors="pt", padding=True)

with torch.no_grad():
    # transformers >= 5: mismo caso que get_image_features — .pooler_output
    vectores_texto = modelo.get_text_features(**entradas_texto).pooler_output

print(f"Shape: {vectores_texto.shape}  ({len(textos)} frases, 512 dimensiones cada una)")

## 4 — ¿Qué tan parecido es el texto a la imagen?

Misma similitud coseno de siempre. Esperamos que `"a photo of a cat"` sea la frase más parecida al vector de la foto del gato.

In [ ]:
def similitud_coseno(u, v):
    u, v = u.detach().numpy(), v.detach().numpy()
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"{'frase':>28s}    sim_coseno")
for texto, vec_texto in zip(textos, vectores_texto):
    sim = similitud_coseno(vector_gato, vec_texto)
    print(f"  {texto!r:>26s}:  {sim:+.3f}")

## 5 — Clasificación zero-shot

"Zero-shot" = el modelo nunca fue entrenado para distinguir estas categorías puntuales, pero puede **comparar** la imagen contra cualquier frase que se nos ocurra. Embebemos las 8 fotos y las 7 frases, armamos la matriz de similitud, y por cada imagen vemos **las dos frases más parecidas** (no solo la ganadora) — con las parejas bicicleta/motocicleta y pizza/hamburguesa, el margen entre el 1er y 2do lugar es lo interesante.

`paisaje de montaña` no tiene una frase correcta en la lista. Un clasificador zero-shot **siempre** elige algo — no existe la opción "ninguna de las anteriores" — así que hay que fijarse en el número de similitud, no solo en cuál ganó. Y para `hot dog`, checa si le gana `"a photo of a hot dog"` o si el modelo se confunde con `"a photo of a dog"`.

In [ ]:
ESPERADO = {
    "gato": "a photo of a cat",
    "perro": "a photo of a dog",
    "bicicleta": "a photo of a bicycle",
    "motocicleta": "a photo of a motorcycle",
    "pizza": "a photo of a pizza",
    "hamburguesa": "a photo of a hamburger",
    "hot dog (comida)": "a photo of a hot dog",
    "paisaje de montaña": None,  # a propósito: ninguna frase de la lista la describe
}

entradas_imgs = procesador(images=list(fotos.values()), return_tensors="pt")
with torch.no_grad():
    vectores_imgs = modelo.get_image_features(**entradas_imgs).pooler_output

# Matriz de similitud: filas = imágenes, columnas = frases
matriz = np.zeros((len(fotos), len(textos)))
for i, vi in enumerate(vectores_imgs):
    for j, vt in enumerate(vectores_texto):
        matriz[i, j] = similitud_coseno(vi, vt)

nombres_fotos = list(fotos.keys())
print(f"{'imagen':>20s}   {'1er lugar':>28s} sim    {'2do lugar':>28s} sim")
for i, nombre in enumerate(nombres_fotos):
    orden = np.argsort(-matriz[i])
    j1, j2 = orden[0], orden[1]
    esperado = ESPERADO[nombre]
    if esperado is None:
        marca = "—"  # no había frase correcta para elegir
    else:
        marca = "✓" if textos[j1] == esperado else "✗"
    print(
        f"{nombre:>20s}   {textos[j1]!r:>28s} {matriz[i, j1]:.3f} {marca}"
        f"  {textos[j2]!r:>28s} {matriz[i, j2]:.3f}"
    )

In [ ]:
azul = plt.cm.colors.LinearSegmentedColormap.from_list("azul", ["#ffffff", "#2a78d6"])

fig, ax = plt.subplots(figsize=(9, 6.5))
im = ax.imshow(matriz, cmap=azul, vmin=0, vmax=matriz.max())

ax.set_xticks(range(len(textos)))
ax.set_xticklabels(textos, rotation=30, ha="right")
ax.set_yticks(range(len(nombres_fotos)))
ax.set_yticklabels(nombres_fotos)

for i in range(len(nombres_fotos)):
    for j in range(len(textos)):
        color = "white" if matriz[i, j] > matriz.max() * 0.6 else "#333333"
        ax.text(j, i, f"{matriz[i, j]:.2f}", ha="center", va="center", color=color, fontsize=8)

ax.set_title("Similitud coseno: imagen × frase")
fig.colorbar(im, ax=ax, label="similitud coseno")
plt.tight_layout()
plt.show()

## Cierre

Con el mismo truco de siempre (vectores + similitud coseno) comparamos ahora **modalidades distintas**: una foto contra una frase. Esa es la idea detrás de la búsqueda "texto → imagen" (como en Google Fotos o Pinterest) y de los modelos que generan imágenes a partir de texto.

**Lo que se ve con este set de 8 imágenes:**

- En las parejas parecidas (bicicleta/motocicleta, pizza/hamburguesa), el 2do lugar suele quedar cerca del 1ro — CLIP las distingue, pero con menos margen que al gato o al perro.
- `hot dog` es la trampa: si el modelo solo mirara la palabra "dog" en el texto, se confundiría con `"a photo of a dog"`. Si mira la imagen de verdad, no.
- `paisaje de montaña` no tenía frase correcta — vale la pena ver qué eligió y qué tan baja quedó esa similitud comparada con las demás filas.

**Límites de esta primera versión:**

- CLIP funciona mejor con texto en **inglés** — en español la similitud baja un poco.
- No hay negative sampling ni entrenamiento aquí: CLIP ya viene entrenado, solo lo usamos.